---

# Semantic Segmentation Pipeline with DINOv3 ViT-L

*A dataset-agnostic binary and multiclass segmentation pipeline built on the DINOv3 ViT-L/16 backbone: a frozen-backbone linear head baseline, a multi-layer convolutional decoder with partial backbone fine-tuning, ensembling, post-processing tuned on out-of-fold masks, an honest holdout score, and export of masks and run-length encoded submissions to `pipeline-output/`.*

---

## Team Name

- Name 1 (Role)
- Name 2
- Name 3

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Toolkit**](#3)
4. [**Data Overview and Decisions**](#4)
5. [**Data Preparation**](#5)
6. [**DINOv3 Backbone**](#6)
7. [**Baseline: Frozen Backbone + Linear Head**](#7)
8. [**Fine-Tuning: Multi-Layer Decoder**](#8)
9. [**Ensemble and Post-Processing**](#9)
10. [**Final Evaluation on Holdout**](#10)
11. [**Model Interpretation**](#11)
12. [**Error Analysis**](#12)
13. [**Predictions and Export**](#13)


---

# Introduction <a name="1"></a>

---

## Overview

*DINOv3 is a self-supervised vision transformer from Meta AI whose patch features are dense: every 16 x 16 patch token describes what is at that location, which is why a linear layer on frozen DINOv3 patches already produces usable segmentation maps. This notebook starts from exactly that baseline and then adds a light convolutional decoder that fuses several transformer layers with a high-resolution image branch, fine-tuning the last blocks only when it pays off on the same folds.*

## Aim

*The notebook pairs images with masks (mask files or run-length encoded tables), trains a linear head on frozen patch features as the baseline, trains the multi-layer decoder with the last transformer blocks unfrozen, blends the two, tunes the threshold and the minimum object size on out-of-fold masks, evaluates once on a held-out set, and writes PNG masks plus a submission that follows `sample_submission.csv`. It supports binary and multiclass semantic segmentation.*

## How to Configure

*Everything is set in the `Settings` class (section 2). Settings left on `'auto'` are computed from the data in section 4 and recorded in `decisions.csv` with their reason.*

| Setting | Purpose | Example |
|---|---|---|
| `KAGGLE_PATH` / `COLAB_PATH` / `LOCAL_PATH` | Dataset folder; images, masks, tables, and the sample submission are found by name | `'/kaggle/input/comp'` |
| `TEST_*_PATH` | Optional test image folder or test table outside the dataset folder | `None` |
| `MASK_DIR`, `MASK_SUFFIX` | Mask folder and file-name suffix when the automatic pairing guesses wrong | `'train/masks'`, `'_mask'` |
| `RLE_COL`, `IMAGE_COL`, `CLASS_COL`, `RLE_ORDER` | Run-length encoded masks in a table; `RLE_ORDER` is `'F'` (top-to-bottom, Kaggle default) or `'C'` (left-to-right) | `'EncodedPixels'`, `'ImageId'`, `'ClassId'`, `'auto'` |
| `MASK_VALUES`, `IGNORE_INDEX`, `CLASS_NAMES` | Raw mask value (or RGB colour) to class index, the value to ignore, readable class names | `{0: 0, 255: 1}`, `255`, `['bg', 'road']` |
| `HF_TOKEN`, `BACKBONE`, `BACKBONE_FALLBACK` | Gated DINOv3 access and the backbone choice, as in the classification notebook | `None`, `'dinov3-vitl16'` |
| `IMG_SIZE` | Square training and inference resolution (rounded to the patch size) | `'auto'`, `448`, `512` |
| `FINETUNE`, `UNFREEZE_LAST_N`, `DECODER_LAYERS` | Train the decoder stage; blocks unfrozen; transformer layers fused by the decoder | `'auto'`, `4`, `'auto'` |
| `EPOCHS`, `LR`, `HEAD_LR`, `LLRD`, `BATCH_SIZE`, `TRAIN_FOLDS` | Fine-tuning budget and folds | `15`, `2e-5`, `5e-4`, `0.8`, `8`, `[0]` |
| `DICE_WEIGHT`, `CLASS_WEIGHTS` | Dice term added to the cross-entropy; class weighting | `'auto'` |
| `TUNE_POSTPROCESS` | Tune the binary threshold and the minimum object size on out-of-fold masks | `True` |
| `EMPTY_RLE` | What an empty mask is written as in the submission | `''`, `'-1'` |

*The backbone registry is the same as in the classification notebook: `dinov3-vitl16` (default), `dinov3-vitl16-sat` (satellite imagery), `dinov3-vith16plus`, `dinov3-vitb16`, `dinov3-vits16`, and the ungated `dinov2-large` / `dinov2-small` fallbacks.*

## Metric

***Dice** (averaged per image, with an empty prediction on an empty mask counted as 1) is the default for binary segmentation, as in most Kaggle segmentation competitions. **Mean IoU** over classes, accumulated over all pixels, is the default for multiclass segmentation. Per-image IoU, dataset-level foreground IoU, mean Dice, and pixel accuracy are reported alongside.*

$$\text{Dice} = \frac{2\,|P \cap G|}{|P| + |G|} \qquad \text{mIoU} = \frac{1}{K}\sum_{k=1}^{K} \frac{|P_k \cap G_k|}{|P_k \cup G_k|}$$

## Dataset

*Any segmentation dataset works. Put the files in `data/` next to this notebook, or point `KAGGLE_PATH` / `COLAB_PATH` / `LOCAL_PATH` at the dataset folder. Two layouts are recognised automatically.*

*Mask files, paired with images by file name (a suffix such as `_mask` is stripped):*

```
data/
├── train/
|    ├── images/   (*.jpg, *.png, *.tif)
|    └── masks/    (*.png, *.gif, *.tif: class values, 0/255, or RGB colours)
└── test/
     └── images/
```

*Run-length encoded masks in a table (one row per image, or per image and class):*

```
data/
├── train.csv              (image id, [class id], RLE)
├── sample_submission.csv  (image id, [class id], RLE)
├── train_images/
└── test_images/
```

*Folders whose name contains `mask`, `label`, `annot`, `gt`, or `seg` hold masks. Mask values are mapped to classes automatically: two values give a binary task, 255 next to small class values is treated as the ignore label (Pascal VOC style), and RGB colour masks get one class per colour. `MASK_VALUES` overrides the mapping whenever the guess is wrong.*

## Approach: Linear Head to Multi-Layer Decoder

*A holdout set is separated first and touched only once at the end. Images with identical bytes are grouped so they never straddle a fold. Predictions are stored as probability maps at the model resolution, so blending and post-processing are tuned without re-running the network.*

```
Images + masks -> value mapping -> duplicate hashing -> holdout split -> grouped CV folds
    |
    +-- Baseline : frozen DINOv3 last-layer patches -> BatchNorm -> 1x1 conv -> upsample
    |               (the DINO linear segmentation probe)          -> pipeline-output/baseline/
    +-- Decoder  : DINOv3 (last N blocks trainable, LLRD), 4 layers fused at 1/16
    |               -> conv -> upsample to 1/8 -> concat CNN detail branch at 1/4 -> conv -> upsample
    |               CE / BCE + Dice, augmentation, mixed precision, early stopping, flip TTA
    +-- Ensemble : probability blend, threshold and minimum object size tuned on OOF masks
         |
    Best by CV -> holdout evaluation -> overlays and per-class IoU -> worst images
         -> pipeline-output/final/ (masks/*.png, submission.csv with RLE, report)
```

---

# Initialization <a name="2"></a>

---

## Environment Setup

The frozen-backbone baseline runs on any device, but the decoder stage with ViT-L needs a GPU. On Kaggle, enable a T4 x2 or P100 accelerator and turn Internet on for the first download. The DINOv3 weights are gated: accept the license once on the model page at huggingface.co (for example `facebook/dinov3-vitl16-pretrain-lvd1689m`), create a read token, and store it as a Kaggle or Colab secret named `HF_TOKEN` (or the `HF_TOKEN` environment variable locally). Without access, the notebook falls back to the ungated `BACKBONE_FALLBACK`.

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook. DINOv3 needs `transformers` 4.56 or newer.

In [ ]:
%pip install -q numpy pandas scipy scikit-learn matplotlib seaborn pillow pyarrow openpyxl torch torchvision "transformers>=4.56" huggingface_hub

## Import Libraries

The secret readers for Kaggle and Colab are imported defensively, because each one exists only on its own platform. `scipy.ndimage` provides the connected components used by post-processing.

In [ ]:
import gc
import os
import re
import json
import math
import time
import random
import hashlib
import warnings
import contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
import seaborn as sns
from PIL import Image, ImageFile, UnidentifiedImageError
from scipy import ndimage
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import tv_tensors
from torchvision.transforms import v2 as T
import transformers
from transformers import AutoConfig, AutoImageProcessor, AutoModel, get_cosine_schedule_with_warmup

try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    UserSecretsClient = None
try:
    from google.colab import userdata as colab_userdata
except ImportError:
    colab_userdata = None

warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()
ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} x {torch.cuda.device_count()}')
print(f'Device: {DEVICE} | torch {torch.__version__} | transformers {transformers.__version__}')

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

## Settings

Paths, mask layout, validation, backbone, baseline, decoder, and submission options are centralised here. `'auto'` values are resolved in section 4 and recorded in the decision table.

In [ ]:
class Settings:
    SEED       = 42
    _ON_KAGGLE = Path('/kaggle/input').exists()
    _ON_COLAB  = Path('/content').exists() and not _ON_KAGGLE

    # 1) Data location: a dataset folder (images, masks, tables, sample submission are found by name)
    KAGGLE_PATH      = '/kaggle/input/<dataset-slug>'
    COLAB_PATH       = '/content/drive/MyDrive/<folder>'
    LOCAL_PATH       = 'data'
    TEST_KAGGLE_PATH = None
    TEST_COLAB_PATH  = None
    TEST_LOCAL_PATH  = None
    READ_KWARGS      = {}

    # 2) Mask layout
    MASK_DIR     = 'auto'
    MASK_SUFFIX  = 'auto'
    RLE_COL      = 'auto'
    IMAGE_COL    = 'auto'
    CLASS_COL    = 'auto'
    RLE_ORDER    = 'auto'
    RLE_INCLUDE_UNLISTED = 'auto'
    MASK_VALUES  = 'auto'
    IGNORE_INDEX = 'auto'
    CLASS_NAMES  = None
    DEBUG_SAMPLE = None

    # 3) Validation
    N_FOLDS          = 5
    HOLDOUT_SIZE     = 0.15
    METRIC           = 'auto'
    SCORE_MAX_SIDE   = 768
    TUNE_POSTPROCESS = True

    # 4) Backbone
    HF_TOKEN          = None
    BACKBONE          = 'dinov3-vitl16'
    BACKBONE_FALLBACK = 'dinov2-large'
    BACKBONE_REGISTRY = {
        'dinov3-vitl16':     'facebook/dinov3-vitl16-pretrain-lvd1689m',
        'dinov3-vitl16-sat': 'facebook/dinov3-vitl16-pretrain-sat493m',
        'dinov3-vith16plus': 'facebook/dinov3-vith16plus-pretrain-lvd1689m',
        'dinov3-vitb16':     'facebook/dinov3-vitb16-pretrain-lvd1689m',
        'dinov3-vits16':     'facebook/dinov3-vits16-pretrain-lvd1689m',
        'dinov2-large':      'facebook/dinov2-large',
        'dinov2-small':      'facebook/dinov2-small',
    }
    IMG_SIZE = 'auto'

    # 5) Baseline: frozen backbone + linear head
    BASELINE        = True
    BASELINE_EPOCHS = 6
    BASELINE_LR     = 1e-3

    # 6) Fine-tuning: multi-layer decoder
    FINETUNE        = 'auto'
    UNFREEZE_LAST_N = 'auto'
    DECODER_LAYERS  = 'auto'
    DECODER_DIM     = 256
    EPOCHS          = 15
    BATCH_SIZE      = 8
    GRAD_ACCUM      = 1
    LR              = 2e-5
    HEAD_LR         = 5e-4
    LLRD            = 0.8
    WEIGHT_DECAY    = 0.05
    WARMUP_RATIO    = 0.05
    DICE_WEIGHT     = 'auto'
    CLASS_WEIGHTS   = 'auto'
    DROP_PATH       = 0.1
    AUGMENT         = 'medium'
    VFLIP           = False
    PATIENCE        = 4
    MIXED_PRECISION = 'auto'
    GRAD_CHECKPOINT = 'auto'
    NUM_WORKERS     = 'auto'
    TRAIN_FOLDS     = [0]
    TTA             = True
    SAVE_WEIGHTS    = False
    ENSEMBLE        = True

    # 7) Submission
    EMPTY_RLE  = ''
    SAVE_MASKS = True

    # 8) Outputs
    DATA_PATH    = KAGGLE_PATH if _ON_KAGGLE else COLAB_PATH if _ON_COLAB else LOCAL_PATH
    TEST_PATH    = TEST_KAGGLE_PATH if _ON_KAGGLE else TEST_COLAB_PATH if _ON_COLAB else TEST_LOCAL_PATH
    OUTPUT_DIR   = Path('/kaggle/working/pipeline-output') if _ON_KAGGLE else Path('pipeline-output')
    BASELINE_DIR = OUTPUT_DIR / 'baseline'
    FINAL_DIR    = OUTPUT_DIR / 'final'

CFG = Settings()
for d in (CFG.BASELINE_DIR / 'figures', CFG.FINAL_DIR / 'figures', CFG.FINAL_DIR / 'models', CFG.FINAL_DIR / 'masks'):
    d.mkdir(parents=True, exist_ok=True)
env = 'Kaggle' if CFG._ON_KAGGLE else 'Colab' if CFG._ON_COLAB else 'Local'
print(f'Environment : {env}')
print(f'Data path   : {CFG.DATA_PATH}')
print(f'Output dir  : {CFG.OUTPUT_DIR.resolve()}')

## Load Dataset

The dataset folder is resolved with the same search order as the other templates. Every image file below it is split into masks (files in a folder named like `masks` / `labels` / `gt`, or with a suffix such as `_mask`) and images. When a train table has a column of run-length encoded strings, the table layout is used: rows are matched to images by full path, file name, or id, rows of the same image are merged, and the class column (when present) says which class each run belongs to. Otherwise each training image is paired with the mask whose name matches after the suffix is stripped. Test images come from a test table, a test folder, or the folder given in `TEST_*_PATH`.

In [ ]:
TABLE_EXT = ('.csv', '.tsv', '.parquet', '.pq', '.feather', '.xlsx', '.xls', '.json', '.jsonl')
IMG_EXT   = ('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tif', '.tiff', '.webp')
SKIP_DIRS = {'eda-output', 'pipeline-output', 'catboost_info', '.ipynb_checkpoints', '.git', '.venv', 'site-packages', '__MACOSX'}
LOCAL_DATA_DIRS = ('data', 'input', 'dataset')
MASK_WORDS = re.compile(r'(mask|label|annot|seg|ground|truth|(?<![a-z])gt(?![a-z]))', re.I)
MASK_SUFFIXES = ('_mask', '-mask', '_masks', '_seg', '_segmentation', '_label', '_labels', '_gt', '_annotation')
RLE_RE = re.compile(r'^\s*\d+\s+\d+(\s+\d+\s+\d+)*\s*$')

def load_table(path, **kwargs):
    suffixes = [s.lower() for s in path.suffixes]
    if '.parquet' in suffixes or '.pq' in suffixes:
        return pd.read_parquet(path, **kwargs)
    if '.feather' in suffixes:
        return pd.read_feather(path, **kwargs)
    if '.xlsx' in suffixes or '.xls' in suffixes:
        return pd.read_excel(path, **kwargs)
    if '.jsonl' in suffixes:
        return pd.read_json(path, lines=True, **kwargs)
    if '.json' in suffixes:
        return pd.read_json(path, **kwargs)
    if '.tsv' in suffixes:
        kwargs.setdefault('sep', '\t')
    return pd.read_csv(path, low_memory=False, **kwargs)

def file_role(f):
    stem = f.name.lower().split('.')[0]
    if 'sample' in stem or 'submission' in stem:
        return 'sample'
    if stem.startswith('test'):
        return 'test'
    if stem.startswith('train'):
        return 'train'
    return None

def folder_role(f, root):
    parts = [p.lower() for p in f.relative_to(root).parts[:-1]] if f.is_relative_to(root) else []
    if any(p.startswith('test') for p in parts):
        return 'test'
    if any(p.startswith('train') for p in parts):
        return 'train'
    return None

def walk(root):
    # os.walk prunes skipped and hidden folders without a disk call per file (slow on the /kaggle/input network mount)
    out = []
    for d, dirs, names in os.walk(root):
        dirs[:] = [x for x in dirs if x not in SKIP_DIRS and not x.startswith('.')]
        out += [Path(d) / n for n in names if not n.startswith('.')]
    return sorted(out)

def find_root(data_path):
    p = Path(data_path) if data_path else None
    if p is not None and p.exists():
        return p
    if CFG._ON_KAGGLE:
        roots = [Path('/kaggle/input')]
    elif CFG._ON_COLAB:
        roots = [Path('/content')]
    else:
        roots = [Path(d) for d in LOCAL_DATA_DIRS] + [Path('.')]
    if p is not None:
        print(f'[warn] {p} not found; searching {[str(r) for r in roots if r.exists()]}')
    for r in (r for r in roots if r.exists()):
        if any(f.suffix.lower() in IMG_EXT for f in walk(r)):
            return r
    return None

def strip_suffix(stem):
    s = stem.lower()
    sufs = [CFG.MASK_SUFFIX.lower()] if CFG.MASK_SUFFIX not in ('auto', None) else MASK_SUFFIXES
    for suf in sorted(sufs, key=len, reverse=True):
        if s.endswith(suf) and len(s) > len(suf):
            return s[:-len(suf)]
    return s

def is_mask(f):
    if MASK_DIR is not None:
        return MASK_DIR in f.parents
    return bool(MASK_WORDS.search(f.parent.name)) or strip_suffix(f.stem) != f.stem.lower()

class ImageIndex:
    def __init__(self, files, root):
        self.root, self.by_name, self.by_stem = Path(root), {}, {}
        for f in files:
            self.by_name.setdefault(f.name.lower(), []).append(f)
            self.by_stem.setdefault(f.stem.lower(), []).append(f)

    def find(self, value, base, prefer=None):
        if isinstance(value, float) and value.is_integer():
            value = int(value)
        v = str(value).strip().replace('\\', '/')
        if not v or v.lower() == 'nan':
            return None
        q = Path(v)
        if q.suffix.lower() in IMG_EXT:
            for cand in (q, Path(base) / q, self.root / q):
                if cand.is_file() and not is_mask(cand):
                    return cand
        hits = self.by_name.get(q.name.lower()) or self.by_stem.get(q.name.lower()) or self.by_stem.get(q.stem.lower())
        if not hits:
            return None
        if len(hits) > 1:
            hits = [h for h in hits if folder_role(h, self.root) == prefer] or hits
            hits = [h for h in hits if '/'.join(h.parts).endswith(v)] or hits
        return hits[0]

def is_text(s):
    return pd.api.types.is_string_dtype(s) or pd.api.types.is_object_dtype(s)

def detect_image_col(table, base, prefer, n=200):
    best, best_key = None, (0.0, False)
    for c in table.columns:
        vals = table[c].dropna().head(n)
        if len(vals) == 0 or not (is_text(vals) or pd.api.types.is_integer_dtype(vals)):
            continue
        rate = float(np.mean([INDEX.find(v, base, prefer) is not None for v in vals]))
        key = (rate, bool(re.search(r'(image|img|file|path|photo|picture)', str(c), re.I)))
        if rate >= 0.5 and key > best_key:
            best, best_key = c, key
    return best

def detect_rle_col(table):
    if CFG.RLE_COL != 'auto':
        return CFG.RLE_COL if CFG.RLE_COL in table.columns else None
    found = []
    for c in table.columns:
        vals = table[c].dropna()
        vals = vals[vals.astype(str).str.strip() != ''].head(200)
        if len(vals) and is_text(vals) and vals.astype(str).str.match(RLE_RE).mean() >= 0.8:
            found.append((bool(re.search(r'(rle|encoded|pixel|mask)', str(c), re.I)), c))
    return max(found)[1] if found else None

def images_frame(files, id_name, sample_ids=None):
    names, stems = [f.name for f in files], [f.stem for f in files]
    use_names = sample_ids is not None and len(sample_ids & set(names)) > len(sample_ids & set(stems))
    return pd.DataFrame({id_name: names if use_names else stems, 'path': files})

ROOT = find_root(CFG.DATA_PATH)
assert ROOT is not None, f'{CFG.DATA_PATH} not found and no folder with images was discovered. Edit the path lines in Settings.'
GIVEN_TABLE = ROOT if ROOT.is_file() else None
ROOT = ROOT.parent if GIVEN_TABLE else ROOT
FILES = walk(ROOT)
TEST_DIR = Path(CFG.TEST_PATH) if CFG.TEST_PATH and Path(CFG.TEST_PATH).is_dir() else None
TEST_TABLE = Path(CFG.TEST_PATH) if CFG.TEST_PATH and Path(CFG.TEST_PATH).is_file() else None
if CFG.TEST_PATH and TEST_DIR is None and TEST_TABLE is None:
    print(f'[warn] test path {CFG.TEST_PATH} not found; searching the dataset folder instead')
if TEST_DIR is not None and not TEST_DIR.is_relative_to(ROOT):
    FILES += walk(TEST_DIR)
MASK_DIR = Path(CFG.MASK_DIR) if CFG.MASK_DIR not in ('auto', None) else None
ALL_IMAGES = [f for f in FILES if f.suffix.lower() in IMG_EXT]
MASK_FILES = [f for f in ALL_IMAGES if is_mask(f)]
IMAGES = [f for f in ALL_IMAGES if not is_mask(f)]
TABLES = [f for f in FILES if f.suffix.lower() in TABLE_EXT]
INDEX = ImageIndex(IMAGES, ROOT)

def pick(role, near=None):
    cands = [f for f in TABLES if file_role(f) == role]
    close = [f for f in cands if near is not None and f.parent == near.parent]
    exact = [f for f in (close or cands) if f.name.lower().split('.')[0] == role]
    return min(exact or close or cands, key=lambda f: len(str(f)), default=None)

RLE_TABLE, RLE_COL, train_tab = None, None, None
candidates = [GIVEN_TABLE] if GIVEN_TABLE else [f for f in TABLES if file_role(f) == 'train'] + [f for f in TABLES if file_role(f) is None]
for f in candidates:
    t = load_table(f, **CFG.READ_KWARGS)
    col = detect_rle_col(t)
    if col is not None:
        RLE_TABLE, RLE_COL, train_tab = f, col, t
        break
SAMPLE_PATH = pick('sample', RLE_TABLE)
SAMPLE = load_table(SAMPLE_PATH) if SAMPLE_PATH else None
SAMPLE_IDS = set(SAMPLE.iloc[:, 0].astype(str)) if SAMPLE is not None else None
RLES, CLASS_COL, LISTED_ONLY_POSITIVE, N_UNLISTED = {}, None, False, 0

if RLE_TABLE is not None:
    LAYOUT = 'rle'
    IMAGE_COL = CFG.IMAGE_COL if CFG.IMAGE_COL != 'auto' else detect_image_col(train_tab, RLE_TABLE.parent, 'train')
    assert IMAGE_COL is not None, f'No column of {RLE_TABLE.name} matches the image files. Set IMAGE_COL.'
    repeated = train_tab[IMAGE_COL].duplicated().any()
    if CFG.CLASS_COL != 'auto':
        CLASS_COL = CFG.CLASS_COL
    elif repeated:
        others = [c for c in train_tab.columns if c not in (IMAGE_COL, RLE_COL) and 2 <= train_tab[c].nunique() <= 50]
        named = [c for c in others if re.search(r'(class|category|label|type)', str(c), re.I)]
        CLASS_COL = (named or others or [None])[0]
    train_tab = train_tab.copy()
    train_tab['path'] = [INDEX.find(v, RLE_TABLE.parent, 'train') for v in train_tab[IMAGE_COL]]
    missing = int(train_tab['path'].isna().sum())
    if missing:
        print(f'[warn] {missing:,} table rows reference an image that was not found; they are dropped')
    train_tab = train_tab[train_tab['path'].notna()]
    rle_str = train_tab[RLE_COL].astype(str).str.strip()
    empty_row = train_tab[RLE_COL].isna() | rle_str.isin(['', 'nan', '-1', '1 0'])
    LISTED_ONLY_POSITIVE = not empty_row.any()
    for p, c, r, e in zip(train_tab['path'], train_tab[CLASS_COL] if CLASS_COL else [None] * len(train_tab), rle_str, empty_row):
        RLES.setdefault(p, [])
        if not e:
            RLES[p].append((c, r))
    include_unlisted = LISTED_ONLY_POSITIVE if CFG.RLE_INCLUDE_UNLISTED == 'auto' else CFG.RLE_INCLUDE_UNLISTED
    if include_unlisted:
        # tables that list only images with objects: the other images of the same folders have empty masks
        folders = {p.parent for p in RLES}
        unlisted = [f for f in IMAGES if f.parent in folders and f not in RLES and folder_role(f, ROOT) != 'test']
        N_UNLISTED = len(unlisted)
        for f in unlisted:
            RLES[f] = []
    raw = pd.DataFrame({'image': [p.name for p in RLES], 'path': list(RLES)})
    raw['mask'] = None
else:
    LAYOUT = 'files'
    IMAGE_COL = 'image'
    by_key = {}
    for m in MASK_FILES:
        by_key.setdefault(strip_suffix(m.stem), []).append(m)
    pairs, unpaired = [], 0
    for f in IMAGES:
        if folder_role(f, ROOT) == 'test' or (TEST_DIR is not None and TEST_DIR in f.parents):
            continue
        cands = by_key.get(f.stem.lower())
        if cands:
            pairs.append((f, max(cands, key=lambda m: len(os.path.commonpath([str(f), str(m)])))))
        else:
            unpaired += 1
    assert pairs, (f'No image could be paired with a mask under {ROOT} ({len(MASK_FILES)} mask files, {len(IMAGES)} images). '
                   'Set MASK_DIR / MASK_SUFFIX, or RLE_COL for a table of run-length encoded masks.')
    if unpaired:
        print(f'[warn] {unpaired:,} non-test images have no mask and are not used for training')
    raw = pd.DataFrame({'image': [f.name for f, _ in pairs], 'path': [f for f, _ in pairs], 'mask': [m for _, m in pairs]})

raw_test = None
TEST_TABLE = TEST_TABLE or (pick('test', RLE_TABLE) if TEST_DIR is None else None)
if TEST_TABLE is not None:
    t = load_table(TEST_TABLE, **CFG.READ_KWARGS)
    col = IMAGE_COL if IMAGE_COL in t.columns else detect_image_col(t, TEST_TABLE.parent, 'test')
    if col is not None:
        t = t.drop_duplicates(col).copy()
        t['path'] = [INDEX.find(v, TEST_TABLE.parent, 'test') for v in t[col]]
        raw_test = t[t['path'].notna()].reset_index(drop=True)
if raw_test is None:
    test_files = [f for f in walk(TEST_DIR) if f.suffix.lower() in IMG_EXT] if TEST_DIR is not None else [f for f in IMAGES if folder_role(f, ROOT) == 'test']
    if test_files:
        raw_test = images_frame(test_files, SAMPLE.columns[0] if SAMPLE is not None else 'id', SAMPLE_IDS)
ID_COL = (SAMPLE.columns[0] if SAMPLE is not None and raw_test is not None and SAMPLE.columns[0] in raw_test.columns
          else 'id' if raw_test is not None and 'id' in raw_test.columns else IMAGE_COL)
if CFG.DEBUG_SAMPLE and len(raw) > CFG.DEBUG_SAMPLE:
    raw = raw.sample(CFG.DEBUG_SAMPLE, random_state=CFG.SEED).reset_index(drop=True)

print(f'Dataset root : {ROOT}')
if LAYOUT == 'rle':
    print(f'Layout       : run-length table {RLE_TABLE.name} (image {IMAGE_COL!r}, RLE {RLE_COL!r}, class {CLASS_COL!r})')
    if N_UNLISTED:
        print(f'               {N_UNLISTED:,} images of the same folders are not in the table and get empty masks')
else:
    print(f'Layout       : mask files ({len(MASK_FILES):,} masks, e.g. {raw["mask"].iloc[0]})')
print(f'Train        : {len(raw):,} images')
print(f'Test         : ' + (f'{len(raw_test):,} images' if raw_test is not None else '(none found: submission.csv will not be written)'))
print(f'Sample sub.  : {SAMPLE_PATH if SAMPLE_PATH else "(none found)"} | id column: {ID_COL!r}')
display(raw.head())

---

# Toolkit <a name="3"></a>

---

The plotting identity matches the other templates, `save_fig` numbers figures per output folder, and `decide` resolves an `'auto'` setting and records the value, its source, and the reason in the decision table exported with the results.

In [ ]:
PRIMARY   = '#3D5A80'
ACCENT    = '#EE6C4D'
SOFT      = '#98C1D9'
DARK      = '#1B263B'
MUTED     = '#8D99AE'
PALETTE   = ['#3D5A80', '#EE6C4D', '#2A9D8F', '#E9C46A', '#9B5DE5',
             '#F15BB5', '#00BBF9', '#8AB17D', '#E76F51', '#264653']
CMAP_SEQ  = LinearSegmentedColormap.from_list('seq', ['#F7F9FC', '#98C1D9', '#3D5A80', '#1B263B'])

sns.set_theme(style='whitegrid', palette=PALETTE)
mpl.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#C9D1DB', 'axes.labelcolor': DARK, 'axes.titleweight': 'bold',
    'axes.titlesize': 12, 'axes.labelsize': 10, 'axes.titlecolor': DARK,
    'xtick.color': '#4A5568', 'ytick.color': '#4A5568',
    'grid.color': '#E2E8F0', 'grid.linewidth': 0.8, 'legend.frameon': False, 'font.size': 10,
})

FIG_COUNTER = {'baseline': 0, 'final': 0}

def slug(text):
    return re.sub(r'[^a-z0-9]+', '_', str(text).lower()).strip('_')[:60]

def save_fig(fig, name, stage='final'):
    FIG_COUNTER[stage] += 1
    root = CFG.BASELINE_DIR if stage == 'baseline' else CFG.FINAL_DIR
    fig.savefig(root / 'figures' / f'{FIG_COUNTER[stage]:02d}_{slug(name)}.png', facecolor='white')
    plt.show()

def load_rgb(path):
    try:
        with Image.open(path) as im:
            return im.convert('RGB')
    except (UnidentifiedImageError, OSError, ValueError):
        return Image.new('RGB', (64, 64))

def image_size(path):
    try:
        with Image.open(path) as im:
            return im.size[1], im.size[0]
    except (UnidentifiedImageError, OSError, ValueError):
        return None

DECISIONS = []

def decide(name, user_value, computed, reason):
    if isinstance(user_value, str) and user_value == 'auto':
        value, source = computed, 'computed'
        why = reason(value) if callable(reason) else reason
    else:
        value, source, why = user_value, 'user', 'set in Settings'
    DECISIONS[:] = [d for d in DECISIONS if d['setting'] != name]
    DECISIONS.append({'setting': name, 'value': str(value)[:80], 'source': source, 'reason': why})
    print(f'{name:<18} = {str(value)[:50]:<22} [{source}] {why}')
    return value

## Run-Length Encoding

Kaggle run-length encoding lists `start length` pairs with 1-based starts. Pixels are numbered either top to bottom then left to right (`'F'`, column-major, the classic Kaggle convention) or left to right then top to bottom (`'C'`, row-major, used by some competitions such as Carvana). Section 4 detects the order by decoding sample masks both ways and keeping the one that gives compact shapes instead of stripes.

In [ ]:
def rle_decode(rle, shape, order='F'):
    s = np.asarray(str(rle).split(), dtype=np.int64)
    flat = np.zeros(shape[0] * shape[1], np.uint8)
    for start, length in zip(s[0::2] - 1, s[1::2]):
        flat[start:start + length] = 1
    return flat.reshape(shape, order=order)

def rle_encode(mask, order='F'):
    pixels = np.concatenate([[0], np.asarray(mask, np.uint8).flatten(order=order), [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(map(str, runs)) if len(runs) else CFG.EMPTY_RLE

def boundary_length(m):
    return int(np.abs(np.diff(m.astype(np.int8), axis=0)).sum() + np.abs(np.diff(m.astype(np.int8), axis=1)).sum())

## Metrics and Post-Processing

Predictions are probability maps with one channel for binary tasks (foreground) and one channel per class for multiclass tasks. `to_mask` turns them into a class map: the binary map is thresholded and connected components smaller than `MIN_AREA` (a fraction of the image area, so it does not depend on resolution) are removed; the multiclass map takes the argmax. Per-image Dice and IoU count an empty prediction on an empty mask as perfect. Mean IoU and mean Dice are accumulated over all pixels of all images, ignoring pixels marked with the ignore label.

In [ ]:
IGNORE = 255
THRESH   = [0.5]
MIN_AREA = [0.0]
LOWER_IS_BETTER = set()

def remove_small(m, frac):
    if frac <= 0 or not m.any():
        return m
    lab, n = ndimage.label(m)
    sizes = ndimage.sum(m, lab, range(1, n + 1))
    keep = np.where(sizes >= frac * m.size)[0] + 1
    return np.isin(lab, keep).astype(np.uint8)

def to_mask(prob, shape=None):
    p = prob.astype(np.float32) / 255.0 if prob.dtype == np.uint8 else prob
    if shape is not None and tuple(p.shape[-2:]) != tuple(shape):
        p = F.interpolate(torch.from_numpy(p)[None], size=tuple(shape), mode='bilinear', align_corners=False)[0].numpy()
    if IS_BIN:
        return remove_small((p[0] >= THRESH[0]).astype(np.uint8), MIN_AREA[0])
    return p.argmax(0).astype(np.uint8)

class SegScore:
    def __init__(self):
        self.cm = np.zeros((N_CLASSES, N_CLASSES), np.int64)
        self.dice, self.iou = [], []

    def add(self, pred, gt):
        valid = gt != IGNORE
        g, p = gt[valid].astype(np.int64), pred[valid].astype(np.int64)
        self.cm += np.bincount(g * N_CLASSES + p, minlength=N_CLASSES ** 2).reshape(N_CLASSES, N_CLASSES)
        gf, pf = g > 0, p > 0
        inter, total, union = (gf & pf).sum(), gf.sum() + pf.sum(), (gf | pf).sum()
        self.dice.append(1.0 if total == 0 else 2 * inter / total)
        self.iou.append(1.0 if union == 0 else inter / union)

    def per_class_iou(self):
        tp = np.diag(self.cm)
        denom = self.cm.sum(0) + self.cm.sum(1) - tp
        return np.where(denom > 0, tp / np.maximum(denom, 1), np.nan)

    def result(self):
        tp = np.diag(self.cm)
        iou = self.per_class_iou()
        dice = np.where(self.cm.sum(0) + self.cm.sum(1) > 0, 2 * tp / np.maximum(self.cm.sum(0) + self.cm.sum(1), 1), np.nan)
        return {'dice': float(np.mean(self.dice)), 'iou': float(np.mean(self.iou)),
                'fg_iou': float(np.nanmean(iou[1:])) if N_CLASSES > 1 else float('nan'),
                'miou': float(np.nanmean(iou)), 'mdice': float(np.nanmean(dice)),
                'pixel_acc': float(tp.sum() / max(self.cm.sum(), 1))}

def better(a, b):
    return b is None or np.isnan(b) or a > b

---

# Data Overview and Decisions <a name="4"></a>

---

This section replaces a separate EDA notebook. It works out how masks encode classes, measures how much of each image the objects cover, checks image sizes and duplicates, and turns the findings into the `'auto'` settings. Each choice is printed with its reason and kept in `DECISIONS`.

## Mask Format and Classes

For mask files, the raw values of up to 200 masks are collected. Two values give a binary task; several small values next to 255 follow the Pascal VOC convention where 255 marks pixels to ignore; many grey levels mean a lossy or anti-aliased binary mask, which is thresholded at half the maximum; RGB masks get one class per distinct frequent colour (colours closer than 48 RGB units are merged, colours are counted on interior pixels only so blended edges are ignored, and the darkest colour is the background), and every pixel is assigned to the nearest colour so that JPEG noise does not create spurious classes. For run-length tables, each value of the class column becomes a class, and the pixel order is detected from the shapes it produces.

In [ ]:
def read_raw_mask(path):
    with Image.open(path) as im:
        if im.mode in ('RGB', 'RGBA'):
            a = np.array(im.convert('RGB'))
            if (a[..., 0] == a[..., 1]).all() and (a[..., 1] == a[..., 2]).all():
                return a[..., 0]
            return a
        if im.mode in ('LA', 'I;16', 'I;16B', 'F'):
            return np.array(im.convert('I') if im.mode != 'LA' else im.convert('L'))
        return np.array(im)

MASK_MODE, VALUE_MAP, IGNORE_RAW, COLOURS, THRESH_VALUE = 'values', {}, None, None, None
if LAYOUT == 'files':
    rng = np.random.default_rng(CFG.SEED)
    probe_masks = list(rng.choice(raw['mask'].values, min(200, len(raw)), replace=False))
    counts, colour = {}, False
    for m in probe_masks:
        a = read_raw_mask(m)
        if a.ndim == 3:
            colour = True
            # only interior pixels (same colour as the right and lower neighbours) count, so blended edges never become classes
            a16 = a.astype(np.int16)
            same = (np.abs(np.diff(a16, axis=0)).sum(-1)[:, :-1] <= 24) & (np.abs(np.diff(a16, axis=1)).sum(-1)[:-1] <= 24)
            u, c = np.unique(a[:-1, :-1][same].reshape(-1, 3), axis=0, return_counts=True)
            keys = [tuple(int(v) for v in row) for row in u]
        else:
            u, c = np.unique(a.astype(np.int64), return_counts=True)
            keys = [int(v) for v in u]
        for k, n in zip(keys, c):
            counts[k] = counts.get(k, 0) + int(n)
    if colour:
        counts = {(k if isinstance(k, tuple) else (k, k, k)): n for k, n in counts.items()}
    total = sum(counts.values())
    keys = sorted(counts)
    if CFG.MASK_VALUES != 'auto':
        VALUE_MAP = {(tuple(k) if isinstance(k, (list, tuple)) else k): int(v) for k, v in CFG.MASK_VALUES.items()}
        MASK_MODE = 'colours' if any(isinstance(k, tuple) for k in VALUE_MAP) else 'values'
        if MASK_MODE == 'colours':
            COLOURS = np.array(list(VALUE_MAP), np.int32)
        why = 'set in Settings'
    elif colour:
        MASK_MODE = 'colours'
        # colours within 48 RGB units of a more frequent one are compression noise: pool them into its cluster first
        centres, pooled = [], []
        for k in sorted(counts, key=counts.get, reverse=True)[:5000]:
            near = [j for j, m in enumerate(centres) if sum((a - b) ** 2 for a, b in zip(k, m)) <= 48 ** 2]
            if near:
                pooled[near[0]] += counts[k]
            else:
                centres.append(k)
                pooled.append(counts[k])
        main = sorted((c for c, n in zip(centres, pooled) if n / total >= 0.001), key=sum)
        COLOURS = np.array(main, np.int32)
        VALUE_MAP = {k: i for i, k in enumerate(main)}
        why = f'{len(main)} RGB colours (frequent, distinct colours; near-duplicates from compression and all other colours map to the nearest one)'
    elif len(keys) > 30:
        MASK_MODE = 'threshold'
        THRESH_VALUE = max(keys) / 2
        VALUE_MAP = {0: 0, 1: 1}
        why = f'{len(keys)} grey levels: lossy binary mask thresholded at {THRESH_VALUE:g}'
    else:
        ignore = CFG.IGNORE_INDEX if CFG.IGNORE_INDEX != 'auto' else (255 if 255 in keys and len(keys) >= 3 else None)
        IGNORE_RAW = ignore
        cls = [k for k in keys if k != ignore]
        if len(cls) <= 2:
            VALUE_MAP = {k: (1 if k == max(cls) and k > 0 else 0) for k in cls}
            if len(cls) == 1 and cls[0] > 0:
                VALUE_MAP = {cls[0]: 1}
        else:
            VALUE_MAP = {k: i for i, k in enumerate(cls)}
        why = f'raw values {keys[:12]}' + (f', {ignore} = ignore' if ignore is not None else '')
    N_CLASSES = max(2, max(VALUE_MAP.values()) + 1)
    default_names = ['background'] + [f'rgb{k}' if MASK_MODE == 'colours' else f'value_{k}' for k, v in sorted(VALUE_MAP.items(), key=lambda kv: kv[1]) if v > 0]
    print(f'Mask encoding: {MASK_MODE} | {why}')
else:
    CLASS_VALUES = sorted({c for v in RLES.values() for c, _ in v}, key=str) if CLASS_COL else [None]
    VALUE_MAP = {c: i + 1 for i, c in enumerate(CLASS_VALUES)}
    N_CLASSES = len(CLASS_VALUES) + 1
    default_names = ['background'] + ([f'class_{c}' for c in CLASS_VALUES] if CLASS_COL else ['foreground'])
    entries = [(p, r) for p, v in RLES.items() for _, r in v][:40]
    votes = {'F': 0, 'C': 0}
    for p, r in entries:
        shape = image_size(p)
        if shape:
            votes['F'] += boundary_length(rle_decode(r, shape, 'F'))
            votes['C'] += boundary_length(rle_decode(r, shape, 'C'))
    order = 'F' if votes['F'] <= votes['C'] else 'C'
    print(f'Run-length tables: {len(entries)} sample masks decoded both ways (boundary length F={votes["F"]:,}, C={votes["C"]:,})')

CLASS_NAMES = list(CFG.CLASS_NAMES) if CFG.CLASS_NAMES else default_names[:N_CLASSES]
TASK = 'binary' if N_CLASSES == 2 else 'multiclass'
IS_BIN = TASK == 'binary'
if LAYOUT == 'rle':
    RLE_ORDER = decide('RLE_ORDER', CFG.RLE_ORDER, order, lambda v: 'column-major (top to bottom), compact shapes' if v == 'F' else 'row-major (left to right), compact shapes')
    decide('RLE_INCLUDE_UNLISTED', CFG.RLE_INCLUDE_UNLISTED, LISTED_ONLY_POSITIVE,
           lambda v: f'the table lists only non-empty masks: {N_UNLISTED:,} unlisted images added as empty' if v else 'the table lists empty masks explicitly')
else:
    RLE_ORDER = CFG.RLE_ORDER if CFG.RLE_ORDER != 'auto' else 'F'
decide('TASK', 'auto', TASK, f'{N_CLASSES} classes including background: {CLASS_NAMES[:8]}')

LUT = None
if LAYOUT == 'files' and MASK_MODE == 'values' and VALUE_MAP:
    top = max(list(VALUE_MAP) + ([IGNORE_RAW] if IGNORE_RAW is not None else []))
    LUT = np.zeros(int(top) + 1, np.uint8)
    for v, k in VALUE_MAP.items():
        LUT[int(v)] = k
    if IGNORE_RAW is not None:
        LUT[int(IGNORE_RAW)] = IGNORE

def map_mask(a):
    if MASK_MODE == 'colours':
        if a.ndim == 2:
            a = np.repeat(a[..., None], 3, -1)
        flat = a.reshape(-1, 3).astype(np.int32)
        out = np.empty(len(flat), np.uint8)
        for s in range(0, len(flat), 1 << 18):
            d = ((flat[s:s + (1 << 18), None, :] - COLOURS[None]) ** 2).sum(-1)
            out[s:s + (1 << 18)] = d.argmin(1)
        return out.reshape(a.shape[:2])
    a = a.astype(np.int64)
    if a.ndim == 3:
        a = a[..., 0]
    if MASK_MODE == 'threshold':
        return (a > THRESH_VALUE).astype(np.uint8)
    return LUT[np.clip(a, 0, len(LUT) - 1)] * (a < len(LUT))

def load_mask(i):
    path = PATHS[i]
    if LAYOUT == 'files':
        return map_mask(read_raw_mask(MASKS[i])).astype(np.uint8)
    shape = image_size(path) or (64, 64)
    m = np.zeros(shape, np.uint8)
    for c, r in RLES[path]:
        m[rle_decode(r, shape, RLE_ORDER) > 0] = VALUE_MAP[c]
    return m

PATHS = raw['path'].tolist()
MASKS = raw['mask'].tolist()
TEST_PATHS = raw_test['path'].tolist() if raw_test is not None else []

## Image Sizes

Image sizes come from the file headers. Unreadable training images are dropped before any mask is decoded.

In [ ]:
sizes = [image_size(p) for p in PATHS]
bad = np.array([s is None for s in sizes])
if bad.any():
    print(f'[warn] {bad.sum():,} unreadable training images dropped: {[str(p) for p in raw.loc[bad, "path"].head(3)]}')
    raw = raw[~bad].reset_index(drop=True)
    PATHS, MASKS = raw['path'].tolist(), raw['mask'].tolist()
    sizes = [s for s in sizes if s is not None]
info = pd.DataFrame(sizes, columns=['height', 'width'])
info['short'] = info[['height', 'width']].min(1)
test_sizes = [image_size(p) or (64, 64) for p in TEST_PATHS]
summary = pd.DataFrame({'train': [len(info), info['height'].median(), info['width'].median(), info['short'].quantile(0.1)]},
                       index=['images', 'median height', 'median width', 'short side p10'])
if test_sizes:
    ts = np.array(test_sizes)
    summary['test'] = [len(ts), np.median(ts[:, 0]), np.median(ts[:, 1]), np.quantile(ts.min(1), 0.1)]
display(summary)

## Mask Statistics

Up to 1,000 training masks are decoded to measure the share of pixels per class, the share of images with no object at all, the foreground coverage per image, and the number of separate objects. Small objects and many empty images favour a Dice term in the loss and post-processing that removes tiny false positives.

In [ ]:
rng = np.random.default_rng(CFG.SEED)
stat_idx = rng.choice(len(raw), min(1000, len(raw)), replace=False)
pix = np.zeros(N_CLASSES, np.int64)
rows = []
mismatch = 0
for i in stat_idx:
    m = load_mask(i)
    valid = m != IGNORE
    pix += np.bincount(m[valid].ravel(), minlength=N_CLASSES)[:N_CLASSES]
    mismatch += m.shape != tuple(sizes[i])
    fg = (m > 0) & valid
    rows.append({'coverage': fg.mean(), 'objects': ndimage.label(fg)[1], 'mask_h': m.shape[0], 'mask_w': m.shape[1],
                 'classes': int(len(np.unique(m[fg])))})
mstats = pd.DataFrame(rows)
PIXEL_SHARE = pix / max(pix.sum(), 1)
EMPTY_SHARE = float((mstats['coverage'] == 0).mean())
FG_SHARE = float(PIXEL_SHARE[1:].sum())
share = pd.DataFrame({'pixel share': PIXEL_SHARE, 'images containing': [np.nan] + [0.0] * (N_CLASSES - 1)}, index=CLASS_NAMES)
for c in range(1, N_CLASSES):
    share.iloc[c, 1] = np.mean([(load_mask(i) == c).any() for i in stat_idx[:200]])
display(share.round(4))
print(f'Images with an empty mask: {EMPTY_SHARE:.1%} | foreground pixels: {FG_SHARE:.2%} | median objects per image: {mstats["objects"].median():.0f}')
if mismatch:
    print(f'[warn] {mismatch} of {len(mstats)} masks differ in size from their image; they are resized to the image (nearest neighbour) when loaded')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(range(N_CLASSES), PIXEL_SHARE, color=[MUTED] + PALETTE[:N_CLASSES - 1] if N_CLASSES <= 11 else PRIMARY)
axes[0].set_xticks(range(N_CLASSES))
axes[0].set_xticklabels([c[:12] for c in CLASS_NAMES], rotation=45 if N_CLASSES > 4 else 0, ha='right' if N_CLASSES > 4 else 'center')
axes[0].set(title='Pixel share per class', yscale='log')
sns.histplot(mstats['coverage'], bins=40, color=PRIMARY, ax=axes[1])
axes[1].set(title='Foreground coverage per image', xlabel='share of image')
sns.histplot(mstats['objects'].clip(upper=30), bins=30, discrete=True, color=PRIMARY, ax=axes[2])
axes[2].set(title='Connected objects per image (clipped at 30)', xlabel='objects')
save_fig(fig, 'mask statistics', 'baseline')

## Sample Overlays

Masks drawn over their images are the quickest check that pairing, value mapping, and run-length order are right: the coloured regions must sit exactly on the objects.

In [ ]:
CLASS_CMAP = ListedColormap([(0, 0, 0, 0)] + [mpl.colors.to_rgba(PALETTE[(k - 1) % len(PALETTE)], 0.55) for k in range(1, max(N_CLASSES, 2))])

def overlay(ax, img, m, title=''):
    ax.imshow(img)
    shown = np.where(m == IGNORE, 0, m)
    ax.imshow(np.ma.masked_equal(shown, 0), cmap=CLASS_CMAP, vmin=0, vmax=max(N_CLASSES - 1, 1), interpolation='nearest')
    ax.set_title(title, fontsize=8)
    ax.axis('off')

cov_of = dict(zip(stat_idx, mstats['coverage']))
with_fg = [i for i in stat_idx if cov_of[i] > 0][:6]
shown_idx = with_fg + [i for i in stat_idx if i not in with_fg][:8 - len(with_fg)]
fig, axes = plt.subplots(2, len(shown_idx), figsize=(len(shown_idx) * 2.3, 4.8))
for j, i in enumerate(shown_idx):
    img, m = load_rgb(PATHS[i]), load_mask(i)
    if m.shape != (img.size[1], img.size[0]):
        m = np.array(Image.fromarray(m).resize(img.size, Image.NEAREST))
    axes[0, j].imshow(img)
    axes[0, j].set_title(str(raw['image'].iloc[i])[:18], fontsize=8)
    axes[0, j].axis('off')
    overlay(axes[1, j], img, m, ', '.join(CLASS_NAMES[c][:10] for c in np.unique(m) if 0 < c < N_CLASSES)[:24] or 'empty')
fig.suptitle('Training images (top) and their masks (bottom)', fontweight='bold')
save_fig(fig, 'sample overlays', 'baseline')

## Duplicates

Exact duplicate images are found by hashing their bytes and are grouped in the splits, so the same picture never sits in both a training fold and its validation fold.

In [ ]:
def file_hash(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

raw['hash'] = [file_hash(p) for p in PATHS]
dup = raw.groupby('hash').size()
print(f'Duplicate groups in train: {int((dup > 1).sum()):,} ({int(dup[dup > 1].sum()):,} images)')
if TEST_PATHS:
    print(f'Test images also in train: {int(pd.Series([file_hash(p) for p in TEST_PATHS]).isin(set(raw["hash"])).sum()):,}')

## Resolved Settings

The input resolution follows the typical short side of the images, capped at 512 on a GPU and 224 without one, because dense prediction benefits more from resolution than classification does. The Dice weight grows when objects are small, and classes whose pixels are rare get a larger cross-entropy weight (square-root inverse frequency, which avoids extreme weights). Decoder layers are spread evenly through the backbone, as in DPT-style heads.

In [ ]:
METRIC = decide('METRIC', CFG.METRIC, 'dice' if IS_BIN else 'miou', lambda v: f'default for {TASK} segmentation')
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == 'cuda' else 0.0
short_side = float(info['short'].median())
size = int(min(512, max(224, short_side))) if DEVICE == 'cuda' else 224
IMG_SIZE = decide('IMG_SIZE', CFG.IMG_SIZE, size, lambda v: f'median short side {short_side:.0f}px on {DEVICE}')
DICE_WEIGHT = decide('DICE_WEIGHT', CFG.DICE_WEIGHT, 1.0 if FG_SHARE < 0.1 else 0.5,
                     f'foreground covers {FG_SHARE:.1%} of pixels')
rare = PIXEL_SHARE[PIXEL_SHARE > 0]
skew = float(rare.max() / rare.min()) if len(rare) else 1.0
CLASS_WEIGHTS = decide('CLASS_WEIGHTS', CFG.CLASS_WEIGHTS, 'sqrt' if (not IS_BIN and skew > 20) else None,
                       f'largest / smallest class pixel share = {skew:.0f}x' if not IS_BIN else 'binary task: Dice handles imbalance')
FINETUNE = decide('FINETUNE', CFG.FINETUNE, DEVICE == 'cuda',
                  'a GPU is available' if DEVICE == 'cuda' else 'no CUDA GPU: the decoder stage would be too slow, the frozen linear head is used')
UNFREEZE_LAST_N = decide('UNFREEZE_LAST_N', CFG.UNFREEZE_LAST_N, 4 if len(raw) < 1000 else 8, f'{len(raw):,} training images')
big = DEVICE == 'cuda' and GPU_GB < 24 and IMG_SIZE >= 448
GRAD_CHECKPOINT = decide('GRAD_CHECKPOINT', CFG.GRAD_CHECKPOINT, bool(big),
                         lambda v: 'saves memory at high resolution on a small GPU' if v else 'not needed')
amp = 'no' if DEVICE != 'cuda' else 'bf16' if torch.cuda.is_bf16_supported() else 'fp16'
MIXED_PRECISION = decide('MIXED_PRECISION', CFG.MIXED_PRECISION, amp, f'{DEVICE} device')
AMP_DTYPE = {'bf16': torch.bfloat16, 'fp16': torch.float16}.get(MIXED_PRECISION)
workers = min(4, os.cpu_count() or 1) if DEVICE == 'cuda' else 0
NUM_WORKERS = decide('NUM_WORKERS', CFG.NUM_WORKERS, workers, 'parallel loading on a GPU machine' if workers else 'in-process loading (safe in notebooks on CPU and Apple silicon)')

def amp_ctx():
    return torch.autocast('cuda', dtype=AMP_DTYPE) if AMP_DTYPE is not None else contextlib.nullcontext()

display(pd.DataFrame(DECISIONS))

#### Insights

> The decision table is the single place to check before a long run: it shows how the masks were read, the task and metric, the resolution, the loss balance, and how much of the backbone is trained. Any row marked `computed` can be overridden in `Settings`; the overlays above are the visual proof that the mask mapping is right.

---

# Data Preparation <a name="5"></a>

---

## Holdout and Cross-Validation Folds

The holdout set is cut first and stays untouched until section 10. The development set is split into `N_FOLDS` folds shared by both stages, so their out-of-fold masks can be compared and blended. Splits are stratified by the dominant class of each image (0 for empty masks, so empty images are spread evenly) and grouped by duplicate hash.

In [ ]:
def strat_label(i):
    if LAYOUT == 'rle':
        v = RLES[PATHS[i]]
        return VALUE_MAP[v[0][0]] if v else 0
    m = load_mask(i)
    c = np.bincount(m[m != IGNORE].ravel(), minlength=N_CLASSES)[1:N_CLASSES]
    return int(c.argmax() + 1) if c.sum() else 0

STRAT = np.array([strat_label(i) for i in range(len(raw))])
GROUPS = raw['hash'].values

def grouped_split(idx, n_splits, seed):
    try:
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        return list(cv.split(idx, STRAT[idx], GROUPS[idx]))
    except ValueError:
        return list(GroupKFold(n_splits=n_splits).split(idx, STRAT[idx], GROUPS[idx]))

all_idx = np.arange(len(raw))
if CFG.HOLDOUT_SIZE and CFG.HOLDOUT_SIZE > 0:
    dev_pos, hold_pos = grouped_split(all_idx, max(2, int(round(1 / CFG.HOLDOUT_SIZE))), CFG.SEED)[0]
    DEV, HOLD = all_idx[dev_pos], all_idx[hold_pos]
else:
    DEV, HOLD = all_idx, np.array([], dtype=int)
HAS_HOLD = len(HOLD) > 0
FOLD = np.full(len(raw), -1)
for k, (_, va) in enumerate(grouped_split(DEV, CFG.N_FOLDS, CFG.SEED + 1)):
    FOLD[DEV[va]] = k

split = pd.DataFrame({'images': [len(DEV), len(HOLD), len(TEST_PATHS)],
                      'empty masks': [float((STRAT[DEV] == 0).mean()), float((STRAT[HOLD] == 0).mean()) if HAS_HOLD else np.nan, np.nan]},
                     index=['development', 'holdout', 'test'])
split['note'] = [f'{CFG.N_FOLDS} CV folds, duplicates grouped', 'scored once in section 10', 'predicted for submission.csv']
display(split.round(3))

## Dataset and Joint Augmentation

Images and masks go through the same geometric transforms with torchvision's `tv_tensors`, which resize masks with nearest-neighbour interpolation so class ids are never blended. `AUGMENT` controls the strength: `'light'` uses a mild random resized crop and horizontal flip, `'medium'` adds colour jitter and small rotations (pixels rotated in from outside are marked as ignore), and `'strong'` adds a wider crop range and blur. `VFLIP = True` adds vertical flips for imagery without a natural orientation. Evaluation resizes to a square input only.

In [ ]:
def build_transforms(train, augment=None):
    augment = augment or CFG.AUGMENT
    if not train or augment in (None, 'none'):
        return T.Resize((IMG_SIZE, IMG_SIZE), antialias=True)
    scale = {'light': (0.7, 1.0), 'medium': (0.5, 1.0), 'strong': (0.3, 1.0)}[augment]
    aug = [T.RandomResizedCrop(IMG_SIZE, scale=scale, ratio=(0.75, 1.33), antialias=True), T.RandomHorizontalFlip()]
    if CFG.VFLIP:
        aug.append(T.RandomVerticalFlip())
    if augment in ('medium', 'strong'):
        aug += [T.RandomApply([T.ColorJitter(0.3, 0.3, 0.2, 0.03)], p=0.7),
                T.RandomApply([T.RandomAffine(degrees=10, fill={tv_tensors.Image: 0, tv_tensors.Mask: IGNORE})], p=0.5)]
    if augment == 'strong':
        aug.append(T.RandomApply([T.GaussianBlur(5)], p=0.2))
    return T.Compose(aug)

class SegDataset(Dataset):
    def __init__(self, paths, idx=None, train=False, augment=None):
        self.paths, self.idx, self.tf = list(paths), idx, build_transforms(train, augment)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, j):
        img = load_rgb(self.paths[j])
        x = tv_tensors.Image(torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1))
        if self.idx is None:
            x = self.tf(x)
            y = torch.zeros(1, dtype=torch.long)
        else:
            m = load_mask(self.idx[j])
            if m.shape != (img.size[1], img.size[0]):
                m = np.array(Image.fromarray(m).resize(img.size, Image.NEAREST))
            x, y = self.tf(x, tv_tensors.Mask(torch.from_numpy(m.astype(np.int64))))
            y = y.as_subclass(torch.Tensor).long()
        x = T.functional.normalize(x.as_subclass(torch.Tensor).float() / 255.0, MEAN, STD)
        return x, y

def make_loader(idx=None, paths=None, train=False, batch_size=None, augment=None):
    paths = paths if paths is not None else [PATHS[i] for i in idx]
    ds = SegDataset(paths, idx, train, augment)
    bs = batch_size or CFG.BATCH_SIZE
    return DataLoader(ds, batch_size=bs, shuffle=train, drop_last=train and len(ds) > bs,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE == 'cuda')

---

# DINOv3 Backbone <a name="6"></a>

---

## Loading the Weights

The token is read from `Settings.HF_TOKEN`, the `HF_TOKEN` environment variable, a Kaggle secret, or a Colab secret, in that order, and is never printed. `BACKBONE` accepts a registry key, any Hugging Face id, or a local folder (for example a Kaggle dataset that holds a `save_pretrained` copy for offline runs). When the weights cannot be loaded, the notebook switches to `BACKBONE_FALLBACK` and records the switch in the decision table.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def hf_token():
    if CFG.HF_TOKEN:
        return CFG.HF_TOKEN
    tok = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
    readers = [lambda: UserSecretsClient().get_secret('HF_TOKEN') if UserSecretsClient else None,
               lambda: colab_userdata.get('HF_TOKEN') if colab_userdata else None]
    for read in readers:
        if tok:
            break
        try:
            tok = read()
        except Exception:
            tok = None
    return tok or None

HF_TOKEN = hf_token()
print('Hugging Face token:', 'found' if HF_TOKEN else 'not found (needed for gated DINOv3 weights unless they are cached or local)')

def load_backbone(key, drop_path=0.0):
    name = CFG.BACKBONE_REGISTRY.get(key, key)
    config = AutoConfig.from_pretrained(name, token=HF_TOKEN)
    if hasattr(config, 'drop_path_rate'):
        config.drop_path_rate = drop_path
    model = AutoModel.from_pretrained(name, config=config, token=HF_TOKEN)
    try:
        proc = AutoImageProcessor.from_pretrained(name, token=HF_TOKEN)
        mean, std = list(proc.image_mean), list(proc.image_std)
    except Exception:
        mean, std = IMAGENET_MEAN, IMAGENET_STD
    return model, name, mean, std

BACKBONE_KEY = None
for key in [CFG.BACKBONE, CFG.BACKBONE_FALLBACK]:
    if not key:
        continue
    try:
        t0 = time.time()
        frozen, BACKBONE_NAME, MEAN, STD = load_backbone(key)
        BACKBONE_KEY = key
        print(f'Loaded {BACKBONE_NAME} in {time.time() - t0:.0f}s')
        break
    except Exception as e:
        print(f'[warn] could not load {key!r}: {str(e).splitlines()[0][:200]}')
assert BACKBONE_KEY is not None, 'No backbone could be loaded. Check HF_TOKEN, the license acceptance, and Internet access.'
fell_back = BACKBONE_KEY != CFG.BACKBONE
DECISIONS.append({'setting': 'BACKBONE', 'value': BACKBONE_NAME, 'source': 'fallback' if fell_back else 'user',
                  'reason': f'{CFG.BACKBONE} could not be loaded' if fell_back else 'set in Settings'})

frozen.eval().requires_grad_(False)
BB_CFG = frozen.config
PATCH = int(BB_CFG.patch_size)
N_REG = int(getattr(BB_CFG, 'num_register_tokens', 0) or 0)
HIDDEN = int(BB_CFG.hidden_size)
N_LAYERS = int(BB_CFG.num_hidden_layers)
requested = IMG_SIZE
IMG_SIZE = max(PATCH, int(round(IMG_SIZE / PATCH)) * PATCH)
if IMG_SIZE != requested:
    print(f'IMG_SIZE rounded from {requested} to {IMG_SIZE} (a multiple of the {PATCH}px patch)')
GRID = IMG_SIZE // PATCH
even = sorted({max(1, round(N_LAYERS * q)) for q in (0.25, 0.5, 0.75, 1.0)})
DECODER_LAYERS = decide('DECODER_LAYERS', CFG.DECODER_LAYERS, even, lambda v: f'evenly spaced over {N_LAYERS} blocks')
N_OUT = 1 if IS_BIN else N_CLASSES
# probability maps are stored at the model resolution, halved for many classes to bound memory
STORE_SIZE = IMG_SIZE if N_OUT <= 3 else min(IMG_SIZE, max(128, IMG_SIZE // 2))
display(pd.DataFrame({'value': [BACKBONE_NAME, f'{sum(p.numel() for p in frozen.parameters()) / 1e6:.0f}M', N_LAYERS, HIDDEN, PATCH, N_REG,
                                 f'{IMG_SIZE}px -> {GRID}x{GRID} patches', str(DECODER_LAYERS), f'{STORE_SIZE}px', str(MEAN)]},
                      index=['checkpoint', 'parameters', 'transformer blocks', 'hidden size', 'patch size', 'register tokens',
                             'input', 'decoder layers', 'stored prediction size', 'normalisation mean']))

## Dense Patch Features

The backbone returns, per layer, the class token, the register tokens (four in DINOv3, none in DINOv2), and one token per patch in row-major order. `patch_maps` takes the patch tokens of the requested layers, applies the backbone's final layer norm to intermediate layers (the last layer is already normalised), and reshapes each into a `hidden x grid x grid` feature map. These maps are the input of both segmentation heads.

In [ ]:
def final_norm(model):
    return getattr(model, 'norm', None) or getattr(model, 'layernorm', None)

def patch_maps(model, x, layers):
    out = model(pixel_values=x, output_hidden_states=True)
    hs, norm = out.hidden_states, final_norm(model)
    gh, gw = x.shape[-2] // PATCH, x.shape[-1] // PATCH
    maps = []
    for i in layers:
        t = out.last_hidden_state if i >= len(hs) - 1 else (norm(hs[i]) if norm is not None else hs[i])
        p = t[:, 1 + N_REG:]
        maps.append(p.transpose(1, 2).reshape(len(x), -1, gh, gw))
    # a contiguous copy: the transposed token view crashes BatchNorm backward on Apple MPS and is slower on CUDA
    return torch.cat(maps, 1).contiguous()

frozen.to(DEVICE)
with torch.no_grad(), amp_ctx():
    demo_x = make_loader(DEV[:2], batch_size=2).__iter__().__next__()[0].to(DEVICE)
    demo = patch_maps(frozen, demo_x, DECODER_LAYERS)
print(f'Feature maps for {len(DECODER_LAYERS)} layers: {tuple(demo.shape)} (batch, layers x hidden, grid, grid)')
assert demo.shape[-1] == GRID

## Patch Feature Map

Projecting the last-layer patch tokens of a few images onto their first three principal components and showing them as RGB is the standard check for DINO features: regions of the same object or material get the same colour, so the objects to segment should already stand out next to their masks. A noisy map usually means the weights did not load or the normalisation is wrong.

In [ ]:
show = list(rng.choice(DEV, min(6, len(DEV)), replace=False))
with torch.no_grad(), amp_ctx():
    x = make_loader(show, batch_size=len(show)).__iter__().__next__()[0].to(DEVICE)
    tok = patch_maps(frozen, x, [N_LAYERS]).float().cpu().numpy()
flat = tok.transpose(0, 2, 3, 1).reshape(-1, HIDDEN)
comp = PCA(n_components=3, random_state=CFG.SEED).fit_transform(flat)
comp = ((comp - comp.min(0)) / (comp.max(0) - comp.min(0) + 1e-8)).reshape(len(show), GRID, GRID, 3)
fig, axes = plt.subplots(3, len(show), figsize=(len(show) * 2.3, 7))
for j, i in enumerate(show):
    img = load_rgb(PATHS[i]).resize((IMG_SIZE, IMG_SIZE))
    m = np.array(Image.fromarray(load_mask(i)).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST))
    axes[0, j].imshow(img)
    overlay(axes[1, j], img, m)
    axes[2, j].imshow(comp[j], interpolation='nearest')
    for ax in axes[:, j]:
        ax.axis('off')
fig.suptitle('Image, mask, and the first three PCA components of the DINO patch tokens', fontweight='bold')
save_fig(fig, 'patch feature pca', 'baseline')

---

# Baseline: Frozen Backbone + Linear Head <a name="7"></a>

---

## Segmentation Heads

Both stages share one model class. The `'linear'` head is the DINO linear segmentation probe: batch norm and a 1 x 1 convolution on the last-layer patch map, upsampled bilinearly to the input size. The `'conv'` head is a light multi-layer decoder: the patch maps of `DECODER_LAYERS` are concatenated and fused at 1/16 resolution, upsampled to 1/8, then concatenated with a two-layer CNN branch that sees the raw image at 1/4 resolution, which restores the thin boundaries that 16-pixel patches cannot represent. When no backbone block is trainable, the backbone runs without gradients and stays in evaluation mode, which keeps the baseline cheap.

In [ ]:
def conv_bn(cin, cout, stride=1):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, stride, 1, bias=False), nn.BatchNorm2d(cout), nn.GELU())

class SegModel(nn.Module):
    def __init__(self, backbone, layers, kind, dim=256):
        super().__init__()
        self.backbone, self.layers, self.kind, self.frozen = backbone, list(layers), kind, True
        cin = HIDDEN * len(self.layers)
        if kind == 'linear':
            self.bn = nn.BatchNorm2d(cin)
            self.cls = nn.Conv2d(cin, N_OUT, 1)
        else:
            self.fuse = nn.Sequential(nn.Conv2d(cin, dim, 1, bias=False), nn.BatchNorm2d(dim), nn.GELU(), conv_bn(dim, dim))
            self.stem = nn.Sequential(conv_bn(3, 32, 2), conv_bn(32, 64, 2))
            self.up1 = conv_bn(dim, dim // 2)
            self.up2 = nn.Sequential(conv_bn(dim // 2 + 64, dim // 4), conv_bn(dim // 4, dim // 4))
            self.cls = nn.Conv2d(dim // 4, N_OUT, 1)

    def train(self, mode=True):
        super().train(mode)
        if self.frozen:
            self.backbone.eval()
        return self

    def forward(self, x):
        with torch.no_grad() if self.frozen else contextlib.nullcontext():
            f = patch_maps(self.backbone, x, self.layers)
        if self.kind == 'linear':
            out = self.cls(self.bn(f))
        else:
            s = self.stem(x)
            f = self.fuse(f)
            f = self.up1(F.interpolate(f, size=(s.shape[-2] // 2, s.shape[-1] // 2), mode='bilinear', align_corners=False))
            f = self.up2(torch.cat([F.interpolate(f, size=s.shape[-2:], mode='bilinear', align_corners=False), s], 1))
            out = self.cls(f)
        return F.interpolate(out, size=x.shape[-2:], mode='bilinear', align_corners=False)

def find_blocks(model):
    return max((m for m in model.modules() if isinstance(m, nn.ModuleList)), key=len)

def param_groups(model, n_unfreeze, head_lr):
    bb = model.backbone
    bb.requires_grad_(False)
    blocks = find_blocks(bb)
    L = len(blocks)
    n = L if n_unfreeze == 'all' else min(int(n_unfreeze), L)
    groups = []

    def add(params, lr):
        params = list(params)
        for p in params:
            p.requires_grad_(True)
        decay = [p for p in params if p.ndim > 1]
        no_decay = [p for p in params if p.ndim <= 1]
        groups.extend(g for g in ({'params': decay, 'lr': lr, 'weight_decay': CFG.WEIGHT_DECAY},
                                  {'params': no_decay, 'lr': lr, 'weight_decay': 0.0}) if g['params'])

    add([p for name, p in model.named_parameters() if not name.startswith('backbone.')], head_lr)
    for i in range(L - n, L):
        add(blocks[i].parameters(), CFG.LR * CFG.LLRD ** (L - 1 - i))
    if n > 0 and final_norm(bb) is not None:
        add(final_norm(bb).parameters(), CFG.LR)
    if n == L:
        seen = {id(p) for g in groups for p in g['params']}
        add([p for p in bb.parameters() if id(p) not in seen], CFG.LR * CFG.LLRD ** L)
    model.frozen = n == 0
    return groups, n

## Loss, Prediction, and Fast Scoring

The loss is cross-entropy (binary cross-entropy for binary tasks) plus `DICE_WEIGHT` times a soft Dice loss. Cross-entropy gives stable per-pixel gradients, while Dice optimises overlap directly and is insensitive to how much background there is. Pixels with the ignore label contribute to neither. Predictions are stored as 8-bit probability maps at `STORE_SIZE`, averaged over flipped views when `TTA` is on. During training and tuning, masks are scored at that size against nearest-neighbour downsampled ground truth, which is fast; section 10 scores the holdout at the original resolution.

In [ ]:
def make_ce_weight():
    if CLASS_WEIGHTS != 'sqrt' or IS_BIN:
        return None
    w = np.where(PIXEL_SHARE > 0, 1 / np.sqrt(np.maximum(PIXEL_SHARE, 1e-8)), 1.0)
    return torch.tensor(w / w.mean(), dtype=torch.float32, device=DEVICE)

CE_WEIGHT = make_ce_weight()

def seg_loss(z, y):
    z = z.float()
    valid = y != IGNORE
    if IS_BIN:
        logit, t = z[:, 0], (y == 1).float()
        bce = F.binary_cross_entropy_with_logits(logit[valid], t[valid]) if valid.any() else logit.sum() * 0
        p, tv = torch.sigmoid(logit) * valid, t * valid
        dice = 1 - ((2 * (p * tv).sum((1, 2)) + 1) / (p.sum((1, 2)) + tv.sum((1, 2)) + 1)).mean()
        return bce + DICE_WEIGHT * dice
    ce = F.cross_entropy(z, y, weight=CE_WEIGHT, ignore_index=IGNORE)
    p = torch.softmax(z, 1) * valid[:, None]
    oh = F.one_hot(torch.where(valid, y, 0), N_CLASSES).permute(0, 3, 1, 2).float() * valid[:, None]
    inter, den = (p * oh).sum((0, 2, 3)), p.sum((0, 2, 3)) + oh.sum((0, 2, 3))
    present = oh.sum((0, 2, 3)) > 0
    dice = 1 - ((2 * inter + 1) / (den + 1))[present].mean() if present.any() else ce * 0
    return ce + DICE_WEIGHT * dice

def to_prob(z):
    return torch.sigmoid(z) if IS_BIN else torch.softmax(z, 1)

@torch.no_grad()
def predict_probs(model, idx=None, paths=None, tta=True):
    model.eval()
    paths = paths if paths is not None else [PATHS[i] for i in idx]
    out = np.zeros((len(paths), N_OUT, STORE_SIZE, STORE_SIZE), np.uint8)
    pos = 0
    for x, _ in make_loader(paths=paths, batch_size=CFG.BATCH_SIZE):
        x = x.to(DEVICE, non_blocking=True)
        with amp_ctx():
            p = to_prob(model(x).float())
            n_views = 1
            if tta:
                p = p + to_prob(model(x.flip(-1)).float()).flip(-1)
                n_views += 1
                if CFG.VFLIP:
                    p = p + to_prob(model(x.flip(-2)).float()).flip(-2)
                    n_views += 1
            p = p.float() / n_views
        if STORE_SIZE != IMG_SIZE:
            p = F.interpolate(p, size=(STORE_SIZE, STORE_SIZE), mode='bilinear', align_corners=False)
        out[pos:pos + len(x)] = (p.clamp(0, 1) * 255).round().byte().cpu().numpy()
        pos += len(x)
    return out

GT_SMALL = {}

def gt_small(i):
    if i not in GT_SMALL:
        GT_SMALL[i] = np.array(Image.fromarray(load_mask(i)).resize((STORE_SIZE, STORE_SIZE), Image.NEAREST))
    return GT_SMALL[i]

def quick_scores(probs, idx):
    sc = SegScore()
    for p, i in zip(probs, idx):
        sc.add(to_mask(p), gt_small(i))
    return sc.result()

def quick_score(probs, idx):
    return quick_scores(probs, idx)[METRIC]

## Training Loop

`train_fold` trains one stage on one fold. A stage that keeps the backbone frozen reuses the loaded backbone; a stage that unfreezes blocks starts from a fresh copy of the pretrained weights, with layer-wise learning rate decay so block $i$ of $L$ learns at $\text{LR} \cdot \text{LLRD}^{L-1-i}$ and the new head at its own rate. Training uses AdamW, a cosine schedule with warmup, gradient clipping, mixed precision on CUDA, and early stopping on the validation metric. `run_stage` loops over `TRAIN_FOLDS` and collects out-of-fold, holdout, and test probability maps.

In [ ]:
POS_DEV = {i: j for j, i in enumerate(DEV)}

def stage_config(stage):
    if stage == 'linear':
        return dict(kind='linear', layers=[N_LAYERS], unfreeze=0, epochs=CFG.BASELINE_EPOCHS, head_lr=CFG.BASELINE_LR, augment='light')
    return dict(kind='conv', layers=DECODER_LAYERS, unfreeze=UNFREEZE_LAST_N, epochs=CFG.EPOCHS, head_lr=CFG.HEAD_LR, augment=CFG.AUGMENT)

def train_fold(k, stage):
    st = stage_config(stage)
    seed_everything(CFG.SEED + k)
    tr, va = DEV[FOLD[DEV] != k], DEV[FOLD[DEV] == k]
    if st['unfreeze'] == 0:
        backbone = frozen
    else:
        backbone = load_backbone(BACKBONE_KEY, CFG.DROP_PATH)[0]
        if GRAD_CHECKPOINT:
            try:
                backbone.gradient_checkpointing_enable()
            except Exception as e:
                print(f'[warn] gradient checkpointing unavailable: {e}')
    model = SegModel(backbone, st['layers'], st['kind'], CFG.DECODER_DIM).to(DEVICE)
    groups, n_trained = param_groups(model, st['unfreeze'], st['head_lr'])
    trainable = [p for g in groups for p in g['params']]
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    opt = torch.optim.AdamW(groups)
    loader = make_loader(tr, train=True, augment=st['augment'])
    steps = max(1, st['epochs'] * math.ceil(len(loader) / CFG.GRAD_ACCUM))
    sched = get_cosine_schedule_with_warmup(opt, int(CFG.WARMUP_RATIO * steps), steps)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_DTYPE == torch.float16)
    best_s, best_state, wait, history = None, None, 0, []
    print(f'[{stage}] fold {k}: {len(tr):,} train / {len(va):,} valid | {n_trained} of {N_LAYERS} blocks trained | '
          f'{sum(p.numel() for p in trainable) / 1e6:.1f}M trainable parameters')
    for epoch in range(st['epochs']):
        model.train()
        t0, total, seen = time.time(), 0.0, 0
        opt.zero_grad(set_to_none=True)
        for step, (x, y) in enumerate(loader):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with amp_ctx():
                z = model(x)
            loss = seg_loss(z, y) / CFG.GRAD_ACCUM
            scaler.scale(loss).backward()
            if (step + 1) % CFG.GRAD_ACCUM == 0 or step + 1 == len(loader):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()
            total += loss.item() * CFG.GRAD_ACCUM * len(x)
            seen += len(x)
        s = quick_score(predict_probs(model, va, tta=False), va)
        history.append({'stage': stage, 'fold': k, 'epoch': epoch + 1, 'train_loss': total / max(seen, 1), METRIC: s, 'seconds': time.time() - t0})
        flag = ''
        if better(s, best_s):
            best_s, wait, flag = s, 0, ' *'
            best_state = {n: v.detach().to('cpu', copy=True) for n, v in model.state_dict().items()
                          if not n.startswith('backbone.') or n in names}
        else:
            wait += 1
        print(f'  epoch {epoch + 1:>2} | loss {total / max(seen, 1):.4f} | {METRIC} {s:.4f} | {time.time() - t0:.0f}s{flag}')
        if wait >= CFG.PATIENCE:
            print(f'  early stop after {epoch + 1} epochs')
            break
    model.load_state_dict(best_state, strict=False)
    if CFG.SAVE_WEIGHTS:
        torch.save(best_state, CFG.FINAL_DIR / 'models' / f'{stage}_fold{k}.pt')
    result = {'va': va, 'best': best_s, 'history': history, 'oof': predict_probs(model, va, tta=CFG.TTA),
              'hold': predict_probs(model, HOLD, tta=CFG.TTA) if HAS_HOLD else None,
              'test': predict_probs(model, paths=TEST_PATHS, tta=CFG.TTA) if TEST_PATHS else None}
    return model, result

def run_stage(stage):
    oof = np.zeros((len(DEV), N_OUT, STORE_SIZE, STORE_SIZE), np.uint8)
    cov = np.zeros(len(DEV), bool)
    hold_sum, test_sum, n_done, best_model, best_s, history = 0, 0, 0, None, None, []
    for k in CFG.TRAIN_FOLDS:
        model, r = train_fold(k, stage)
        pos = [POS_DEV[i] for i in r['va']]
        oof[pos], cov[pos] = r['oof'], True
        hold_sum = hold_sum + (r['hold'].astype(np.uint16) if r['hold'] is not None else 0)
        test_sum = test_sum + (r['test'].astype(np.uint16) if r['test'] is not None else 0)
        n_done += 1
        history += r['history']
        if better(r['best'], best_s):
            best_s, best_model = r['best'], model.to('cpu')
        else:
            del model
        gc.collect()
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
    avg = lambda s: (s / n_done).round().astype(np.uint8) if isinstance(s, np.ndarray) else None
    print(f'[{stage}] out-of-fold {METRIC} on {cov.sum():,} images: {quick_score(oof[cov], DEV[cov]):.4f}')
    return {'oof': oof, 'cov': cov, 'hold': avg(hold_sum), 'test': avg(test_sum)}, best_model, history

CANDIDATES, MODELS, HISTORY = {}, {}, []

## Linear Head Training

The baseline trains only the batch norm and the 1 x 1 convolution on frozen last-layer patch features, with light augmentation and `BASELINE_EPOCHS` epochs. Because the backbone runs without gradients, this stage is affordable even without a GPU.

In [ ]:
if CFG.BASELINE:
    CANDIDATES['linear'], MODELS['linear'], hist = run_stage('linear')
    HISTORY += hist
else:
    print('Baseline skipped (BASELINE is False).')

## Baseline Results and Export

The baseline's out-of-fold scores, holdout scores (with the default 0.5 threshold), and a complete baseline submission are written to `pipeline-output/baseline/`, so a valid entry exists even if the decoder stage is skipped or interrupted. The submission follows `sample_submission.csv`: one run-length string per image, or per image and class when the sample has a class column (or ids such as `image_3`). Without a sample, binary tasks get `id, rle` and multiclass tasks get `id, class, rle` rows.

In [ ]:
def class_index(v):
    for c, k in VALUE_MAP.items():
        if str(c) == str(v):
            return k
    if str(v) in CLASS_NAMES:
        return CLASS_NAMES.index(str(v))
    try:
        k = int(float(v))
        return k if 0 < k < N_CLASSES else None
    except (TypeError, ValueError):
        return None

def write_submission(test_prob, folder, save_masks=False):
    if raw_test is None or test_prob is None:
        return None
    ids = raw_test[ID_COL].astype(str).tolist()
    masks = {}
    for j, (key, p) in enumerate(zip(ids, raw_test['path'])):
        m = to_mask(test_prob[j], image_size(p) or (STORE_SIZE, STORE_SIZE))
        masks[key] = m
        if save_masks:
            Image.fromarray((m * 255 if IS_BIN else m).astype(np.uint8)).save(CFG.FINAL_DIR / 'masks' / f'{Path(str(p)).stem}.png')
    if SAMPLE is None:
        if IS_BIN:
            sub = pd.DataFrame({ID_COL: ids, 'rle': [rle_encode(masks[i], RLE_ORDER) for i in ids]})
        else:
            sub = pd.DataFrame([{ID_COL: i, 'class': CLASS_NAMES[c], 'rle': rle_encode(masks[i] == c, RLE_ORDER)}
                                for i in ids for c in range(1, N_CLASSES)])
    else:
        cols = list(SAMPLE.columns)
        sid = cols[0]
        rle_col = next((c for c in cols[1:] if re.search(r'(rle|encoded|pixel|mask|segment|predict)', str(c), re.I)), cols[-1])
        class_col = next((c for c in cols[1:] if c != rle_col), None)
        sub, vals, merged = SAMPLE.copy(), [], 0
        for key, cls_v in zip(SAMPLE[sid].astype(str), SAMPLE[class_col] if class_col else [None] * len(SAMPLE)):
            cls = class_index(cls_v) if class_col else None
            if key not in masks and class_col is None and '_' in key:
                base, tail = key.rsplit('_', 1)
                if base in masks:
                    key, cls = base, class_index(tail)
            m = masks.get(key)
            if m is None:
                vals.append(np.nan)
                continue
            if cls is None and not IS_BIN:
                merged += 1
            vals.append(rle_encode(m == cls if cls is not None else m > 0, RLE_ORDER))
        sub[rle_col] = vals
        missing = int(pd.isna(sub[rle_col]).sum())
        if missing:
            print(f'[warn] {missing:,} sample-submission rows have no prediction')
        if merged:
            print(f'[warn] the sample has one row per image, so the {N_CLASSES - 1} foreground classes are merged into one mask')
    sub.to_csv(folder / 'submission.csv', index=False)
    print(f'submission.csv: {sub.shape[0]:,} rows x {sub.shape[1]} cols -> {folder}')
    return sub

SCORE_COLS = ['dice', 'iou', 'fg_iou', 'miou', 'mdice', 'pixel_acc']
if 'linear' in CANDIDATES:
    res = CANDIDATES['linear']
    base_scores = {'cv (out-of-fold)': quick_scores(res['oof'][res['cov']], DEV[res['cov']])}
    if HAS_HOLD:
        base_scores['holdout'] = quick_scores(res['hold'], HOLD)
    base_scores = pd.DataFrame(base_scores).T[SCORE_COLS].round(4)
    display(base_scores)
    base_scores.to_csv(CFG.BASELINE_DIR / 'scores.csv')
    write_submission(res['test'], CFG.BASELINE_DIR)

---

# Fine-Tuning: Multi-Layer Decoder <a name="8"></a>

---

## Decoder Training

The decoder stage fuses the `DECODER_LAYERS` patch maps with the image detail branch and trains the last `UNFREEZE_LAST_N` transformer blocks together with the decoder, for up to `EPOCHS` epochs with `AUGMENT` augmentation. Intermediate layers carry more spatial detail and lower-level cues than the last layer, which mostly encodes semantics; combining them is what lets the decoder outperform the linear probe on boundaries and small objects.

In [ ]:
if FINETUNE:
    if UNFREEZE_LAST_N != 0:
        frozen.to('cpu')
    CANDIDATES['decoder'], MODELS['decoder'], hist = run_stage('decoder')
    HISTORY += hist
else:
    print('Decoder stage skipped (FINETUNE is False). The linear head is the final model.')

## Learning Curves

Training loss and the validation metric per epoch for each stage and fold. A validation curve that peaks early while the loss keeps falling is overfitting: fewer unfrozen blocks, stronger augmentation, or a lower `LR` usually help.

In [ ]:
if HISTORY:
    hist = pd.DataFrame(HISTORY)
    hist.to_csv(CFG.FINAL_DIR / 'training_history.csv', index=False)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for j, ((stage, k), h) in enumerate(hist.groupby(['stage', 'fold'])):
        axes[0].plot(h['epoch'], h['train_loss'], marker='o', color=PALETTE[j % len(PALETTE)], label=f'{stage} fold {k}')
        axes[1].plot(h['epoch'], h[METRIC], marker='o', color=PALETTE[j % len(PALETTE)], label=f'{stage} fold {k}')
    axes[0].set(title='Training loss', xlabel='epoch')
    axes[1].set(title=f'Validation {METRIC}', xlabel='epoch')
    axes[1].legend()
    save_fig(fig, 'learning curves')

---

# Ensemble and Post-Processing <a name="9"></a>

---

## Candidate Comparison and Blend

Candidates are compared on the same development images: those that every candidate has an out-of-fold mask for. The blend averages the linear and decoder probability maps with the weight that scores best on those images. The candidate with the best out-of-fold score becomes the final model.

In [ ]:
assert CANDIDATES, 'No model was trained: enable BASELINE or FINETUNE.'
common = np.logical_and.reduce([c['cov'] for c in CANDIDATES.values()])

def mix(a, b, w):
    if a is None or b is None:
        return None
    return (w * a.astype(np.float32) + (1 - w) * b.astype(np.float32)).round().astype(np.uint8)

if CFG.ENSEMBLE and {'linear', 'decoder'} <= set(CANDIDATES):
    a, b = CANDIDATES['decoder'], CANDIDATES['linear']
    grid = np.round(np.linspace(0, 1, 11), 2)
    blend_scores = [quick_score(mix(a['oof'][common], b['oof'][common], w), DEV[common]) for w in grid]
    w_best = float(grid[int(np.nanargmax(blend_scores))])
    CANDIDATES['blend'] = {'oof': mix(a['oof'], b['oof'], w_best), 'cov': common.copy(), 'hold': mix(a['hold'], b['hold'], w_best),
                           'test': mix(a['test'], b['test'], w_best), 'weight': w_best}
    print(f'Blend weight: {w_best:.1f} x decoder + {1 - w_best:.1f} x linear')

rows = []
for name, c in CANDIDATES.items():
    rows.append({'model': name, f'{METRIC} (common images)': quick_score(c['oof'][common], DEV[common]),
                 f'{METRIC} (own images)': quick_score(c['oof'][c['cov']], DEV[c['cov']]), 'images': int(c['cov'].sum())})
cv_table = pd.DataFrame(rows).set_index('model').round(4)
SELECTED = cv_table[f'{METRIC} (common images)'].idxmax()
FINAL = CANDIDATES[SELECTED]
display(cv_table)
print(f'Selected model: {SELECTED}')

## Threshold and Minimum Object Size

For binary tasks, the probability threshold and the minimum object size (as a fraction of the image area) are tuned jointly on the selected model's out-of-fold masks. Removing tiny components cleans speckle false positives, and on datasets with many empty images it turns near-empty predictions into the empty masks that score 1. The holdout stays untouched, so section 10 shows whether the tuned values generalise. Multiclass tasks use the argmax.

In [ ]:
TUNE_GRID = None
oof_sel, idx_sel = FINAL['oof'][FINAL['cov']], DEV[FINAL['cov']]
before = quick_score(oof_sel, idx_sel)
if IS_BIN and CFG.TUNE_POSTPROCESS:
    rows = []
    for t in np.round(np.linspace(0.2, 0.8, 13), 2):
        for a in (0.0, 0.0005, 0.001, 0.0025, 0.005, 0.01, 0.02):
            THRESH[0], MIN_AREA[0] = t, a
            rows.append({'threshold': t, 'min_area': a, METRIC: quick_score(oof_sel, idx_sel)})
    TUNE_GRID = pd.DataFrame(rows)
    best = TUNE_GRID.loc[TUNE_GRID[METRIC].idxmax()]
    THRESH[0], MIN_AREA[0] = float(best['threshold']), float(best['min_area'])
    piv = TUNE_GRID.pivot(index='min_area', columns='threshold', values=METRIC)
    fig, ax = plt.subplots(figsize=(11, 3.8))
    sns.heatmap(piv, annot=True, fmt='.3f', cmap=CMAP_SEQ, cbar=False, ax=ax, annot_kws={'fontsize': 7})
    ax.set(title=f'Out-of-fold {METRIC} by threshold and minimum object size (share of image)')
    save_fig(fig, 'postprocess tuning')
RULE = f'threshold {THRESH[0]:.2f}, min object {MIN_AREA[0]:.2%} of image' if IS_BIN else 'argmax'
print(f'Post-processing: {RULE}')
print(f'Out-of-fold {METRIC}: {before:.4f} -> {quick_score(oof_sel, idx_sel):.4f}')

---

# Final Evaluation on Holdout <a name="10"></a>

---

## Holdout Scores at Full Resolution

Every candidate is scored once on the holdout set with the tuned post-processing. Predictions are upsampled to each image's original size (capped at `SCORE_MAX_SIDE` on the longer side to bound memory) and compared with the original masks, so these numbers match how a competition scores the submission. The selected model's holdout score is the number to report.

In [ ]:
def full_gt(i):
    m = load_mask(i)
    h, w = m.shape
    scale = min(1.0, CFG.SCORE_MAX_SIDE / max(h, w))
    if scale < 1:
        m = np.array(Image.fromarray(m).resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.NEAREST))
    return m

def full_scores(probs, idx):
    sc = SegScore()
    for p, i in zip(probs, idx):
        gt = full_gt(i)
        sc.add(to_mask(p, gt.shape), gt)
    return sc

HOLD_SCORES, HOLD_SC = None, None
if HAS_HOLD:
    res = {}
    for name, c in CANDIDATES.items():
        if c['hold'] is not None:
            sc = full_scores(c['hold'], HOLD)
            res[f'{name} (selected)' if name == SELECTED else name] = sc.result()
            if name == SELECTED:
                HOLD_SC = sc
    HOLD_SCORES = pd.DataFrame(res).T[SCORE_COLS].round(4)
    display(HOLD_SCORES)
    HOLD_SCORES.to_csv(CFG.FINAL_DIR / 'holdout_scores.csv')
else:
    print('HOLDOUT_SIZE is 0: no holdout evaluation.')

## Per-Class IoU and Pixel Confusion

Per-class IoU shows which classes the model segments well. The row-normalised pixel confusion matrix shows where the errors go: confusion with the background means missed or over-extended objects, confusion between foreground classes means the right region with the wrong label.

In [ ]:
if HOLD_SC is not None:
    iou = HOLD_SC.per_class_iou()
    cm = HOLD_SC.cm / np.maximum(HOLD_SC.cm.sum(1, keepdims=True), 1)
    names = [c[:14] for c in CLASS_NAMES]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={'width_ratios': [1, 1.2]})
    axes[0].barh(names[::-1], np.nan_to_num(iou[::-1]), color=[MUTED if k == 0 else PRIMARY for k in range(N_CLASSES)][::-1])
    axes[0].set(title='IoU per class (holdout)', xlim=(0, 1))
    for y, v in enumerate(iou[::-1]):
        axes[0].text(min(np.nan_to_num(v) + 0.01, 0.9), y, f'{v:.3f}', va='center', fontsize=8)
    sns.heatmap(cm, annot=N_CLASSES <= 10, fmt='.2f', cmap=CMAP_SEQ, xticklabels=names, yticklabels=names, cbar=False, ax=axes[1])
    axes[1].set(title='Pixel confusion (row-normalised)', xlabel='predicted', ylabel='true')
    save_fig(fig, 'per class iou and confusion')

---

# Model Interpretation <a name="11"></a>

---

## Prediction Overlays

Random holdout images with the ground truth, the predicted mask, and an error map: green pixels are correct foreground, red pixels are false positives, and blue pixels are missed foreground. For binary tasks the last column shows the raw probability map, which reveals whether errors come from uncertain regions (values near 0.5) or confident mistakes.

In [ ]:
def error_map(pred, gt):
    err = np.zeros(gt.shape + (4,), np.float32)
    valid = gt != IGNORE
    err[(pred == gt) & (gt > 0) & valid] = (0.16, 0.62, 0.56, 0.6)
    err[(pred != gt) & (pred > 0) & valid] = (0.93, 0.42, 0.30, 0.8)
    err[(pred != gt) & (pred == 0) & (gt > 0) & valid] = (0.24, 0.35, 0.50, 0.8)
    return err

if HAS_HOLD:
    rng_h = np.random.default_rng(CFG.SEED)
    has_fg = [j for j in range(len(HOLD)) if STRAT[HOLD[j]] > 0]
    picks = list(rng_h.choice(has_fg or list(range(len(HOLD))), min(5, len(has_fg or HOLD)), replace=False))
    n_cols = 5 if IS_BIN else 4
    fig, axes = plt.subplots(len(picks), n_cols, figsize=(n_cols * 2.6, len(picks) * 2.6))
    axes = np.atleast_2d(axes)
    for r, j in enumerate(picks):
        i = HOLD[j]
        img, gt = load_rgb(PATHS[i]), load_mask(i)
        if gt.shape != (img.size[1], img.size[0]):
            gt = np.array(Image.fromarray(gt).resize(img.size, Image.NEAREST))
        pred = to_mask(FINAL['hold'][j], gt.shape)
        axes[r, 0].imshow(img)
        axes[r, 0].set_title(str(raw['image'].iloc[i])[:18], fontsize=8)
        overlay(axes[r, 1], img, gt, 'ground truth')
        overlay(axes[r, 2], img, pred, 'prediction')
        axes[r, 3].imshow(img)
        axes[r, 3].imshow(error_map(pred, gt))
        axes[r, 3].set_title('errors', fontsize=8)
        if IS_BIN:
            axes[r, 4].imshow(FINAL['hold'][j][0] / 255.0, cmap=CMAP_SEQ, vmin=0, vmax=1)
            axes[r, 4].set_title('probability', fontsize=8)
        for ax in axes[r]:
            ax.axis('off')
    fig.suptitle(f'Holdout predictions ({SELECTED}); errors: red = false positive, blue = missed', fontweight='bold')
    save_fig(fig, 'holdout prediction overlays')

---

# Error Analysis <a name="12"></a>

---

## Worst Holdout Images and Object Size

Per-image Dice ranks the holdout images from worst to best. The worst ones usually show a systematic weakness: very small objects, thin structures, low contrast, or doubtful labels. The scatter relates per-image Dice to how much of the image the true object covers, which tells whether a higher `IMG_SIZE` (more pixels per small object) is worth its cost.

In [ ]:
if HOLD_SC is not None:
    cover = np.array([float((full_gt(i) > 0).mean()) for i in HOLD])
    per_img = pd.DataFrame({'image': raw['image'].iloc[HOLD].values, 'dice': HOLD_SC.dice, 'iou': HOLD_SC.iou, 'coverage': cover})
    per_img.to_csv(CFG.FINAL_DIR / 'holdout_per_image.csv', index=False)
    worst = per_img.sort_values('dice').index[:8].tolist()
    fig, axes = plt.subplots(2, len(worst), figsize=(len(worst) * 2.3, 5))
    axes = np.atleast_2d(axes.T).T if len(worst) == 1 else axes
    for c, j in enumerate(worst):
        i = HOLD[j]
        img, gt = load_rgb(PATHS[i]), load_mask(i)
        if gt.shape != (img.size[1], img.size[0]):
            gt = np.array(Image.fromarray(gt).resize(img.size, Image.NEAREST))
        pred = to_mask(FINAL['hold'][j], gt.shape)
        overlay(axes[0, c], img, gt, f'truth | dice {per_img.dice[j]:.2f}')
        overlay(axes[1, c], img, pred, 'prediction')
    fig.suptitle('Lowest-Dice holdout images', fontweight='bold')
    save_fig(fig, 'worst holdout images')

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(per_img['coverage'].clip(lower=1e-5), per_img['dice'], s=14, alpha=0.6, color=PRIMARY)
    ax.set(xscale='log', title='Per-image Dice vs object coverage (holdout)', xlabel='share of image covered by the true mask', ylabel='Dice')
    save_fig(fig, 'dice vs coverage')
    print(f'Empty holdout masks: {(cover == 0).sum()} | their mean Dice (1 = correctly empty): {per_img.dice[cover == 0].mean():.3f}')

---

# Predictions and Export <a name="13"></a>

---

## Submission and Artefacts

Test predictions of the selected model are upsampled to each test image's original size, post-processed with the tuned rule, saved as PNG masks in `pipeline-output/final/masks/` (0/255 for binary, class index for multiclass), and run-length encoded into `submission.csv` following `sample_submission.csv`. On Kaggle, `submission.csv` is also copied to `/kaggle/working/`. The decision table and a JSON report complete the export.

In [ ]:
SUB = None
if FINAL['test'] is not None:
    SUB = write_submission(FINAL['test'], CFG.FINAL_DIR, save_masks=CFG.SAVE_MASKS)
    if CFG._ON_KAGGLE:
        SUB.to_csv('/kaggle/working/submission.csv', index=False)
    display(SUB.head())

pd.DataFrame(DECISIONS).to_csv(CFG.FINAL_DIR / 'decisions.csv', index=False)
report = {
    'backbone': BACKBONE_NAME, 'img_size': IMG_SIZE, 'task': TASK, 'metric': METRIC, 'classes': CLASS_NAMES,
    'layout': LAYOUT, 'rle_order': RLE_ORDER, 'decoder_layers': DECODER_LAYERS,
    'unfreeze_last_n': UNFREEZE_LAST_N if FINETUNE else 0,
    'n_train': int(len(raw)), 'n_dev': int(len(DEV)), 'n_holdout': int(len(HOLD)), 'n_test': len(TEST_PATHS),
    'cv': cv_table.reset_index().to_dict('records'), 'selected': SELECTED, 'postprocess': RULE,
    'threshold': THRESH[0], 'min_area': MIN_AREA[0], 'blend_weight_decoder': CANDIDATES.get('blend', {}).get('weight'),
    'holdout': HOLD_SCORES.reset_index().to_dict('records') if HOLD_SCORES is not None else None,
    'decisions': DECISIONS,
}
(CFG.FINAL_DIR / 'report.json').write_text(json.dumps(report, indent=2, default=str))
n_masks = len(list((CFG.FINAL_DIR / 'masks').glob('*.png')))
print(f'Artefacts written to {CFG.FINAL_DIR.resolve()}:')
for f in sorted(CFG.FINAL_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name}')
print(f'  masks/ ({n_masks:,} PNG files)')

#### Insights

> The report records the backbone that actually ran (including a fallback), the resolution and decoder layers, the CV comparison between the linear head, the decoder, and their blend, the tuned post-processing, and the full-resolution holdout score. Together with the overlays, the per-class IoU, and the worst-image grid, they are the material for the written report and the presentation.